# FastOTF2 Converter Benchmark

One-click comparison of the **Chapel** `fastotf2` converter against **Python** and **C**
converters, measuring the wall-clock time to convert OTF2 traces to tabular output.

| Tool | CSV | Parquet |
|------|-----|---------|
| Chapel (`fastotf2` container) | ✓ | ✓ |
| Python (`otf2` + pyarrow) | ✓ | ✓ |
| C (OTF2 C library) | ✓ | — (Arrow toolchain out of scope) |

All three converters ship inside **one Apptainer container**, so they use an identical
OTF2 version and the workflow is portable: the only host requirements are Apptainer + SLURM
and readable trace paths.

**All configuration lives in the config cell below** — there is no external config file.

Traces are labelled by their **measured on-disk size in GiB** (e.g. `11.6GiB`), not by node
count, so every run and every results row is keyed by data size.

## How runs are organised

Each execution of the **config cell (§0)** mints a fresh timestamped `RUN_TAG` and its own
folder under `out/`, so runs never clobber each other:

```
out/run_YYYYMMDD_HHMMSS/
  config.json      # exact parameters used (provenance)
  run.sbatch       # the generated SLURM script that was submitted
  results.csv      # timings for this run
  slurm_logs/      # SLURM stdout/stderr for this run
  run_logs/        # per-conversion converter logs
  scratch/         # conversion output (deleted between runs; large)
  plots/           # rendered table + chart
```

- **New run:** Run All (or run §0→§3). A new `RUN_DIR` is created and submitted.
- **Re-analyse a previous run** (no new data): restart the kernel, run the §0 imports, then
  in §4 set `ANALYZE_RUN = "run_YYYYMMDD_HHMMSS"` (a tag or a path) and run the analysis
  cells. Nothing is resubmitted.
- **Resubmit while another run is going:** the running SLURM job is independent, so you can
  restart this kernel and launch a fresh run at any time — they write to different folders.

> The default trace set here is **~0.7 GiB + ~11.6 GiB** (the 2- and 16-node traces). The
> **~32.7 GiB** (32-node) trace is deliberately excluded so this run does not interfere with
> a separately-running 32-node job; combine results later if needed.


# 0. Configuration & run folder

Everything you might change lives in this one cell. Running it creates a new timestamped
run folder under `out/`. Set `DRY_RUN = True` to preview submission without launching.

**`SYSTEM`** picks which cluster's settings to use (`"frontier"` or `"other-ex"`) — the
trace paths, SLURM partition, Chapel thread count, and `sbatch` account/mail extras all
follow from this one switch via `SYSTEM_CONFIGS`. The Slurm **account** and **mail** address
are secrets and are never hardcoded here — create a gitignored `local_secrets.py` in the
repo root with `SLURM_ACCOUNT` / `SLURM_MAIL_USER` (env vars work too, but only if set
before the Jupyter server process itself starts — an already-running kernel won't pick up a
shell `export` after the fact). On Frontier these become `--account=…` plus
`--mail-type=BEGIN,END,FAIL --mail-user=…` on every job.


In [ ]:
import os, json, shlex, subprocess, time
from pathlib import Path
from datetime import datetime

REPO_DIR = Path.cwd()
assert (REPO_DIR / "benchmark" / "run_one.sh").exists(), \
    f"Run this notebook from the repo root; benchmark/run_one.sh not found in {REPO_DIR}"

# ======================== USER CONFIGURATION ========================
# --- Which system are we running on? ---------------------------------------
# Flip this ONE line to switch clusters; everything system-specific (trace paths,
# SLURM partition, Chapel thread count, and the sbatch account/mail extras) lives in
# SYSTEM_CONFIGS below, so nothing else in the notebook needs to change.
SYSTEM = "frontier"   # "frontier" | "other-ex"

# --- Secrets (never hardcode these in the notebook) -------------------------
# Preferred: a gitignored local_secrets.py in the repo root, since env vars exported in a
# terminal are NOT inherited by an already-running Jupyter kernel/server (exporting them
# and restarting the kernel does nothing -- you'd have to fully relaunch the process that
# started Jupyter). local_secrets.py sidesteps that:
#   SLURM_ACCOUNT   = "csc688"
#   SLURM_MAIL_USER = "you@example.com"
# Falls back to SLURM_ACCOUNT / SLURM_MAIL_USER env vars if that file doesn't exist.
try:
    from local_secrets import SLURM_ACCOUNT, SLURM_MAIL_USER
except ImportError:
    SLURM_ACCOUNT   = os.environ.get("SLURM_ACCOUNT")
    SLURM_MAIL_USER = os.environ.get("SLURM_MAIL_USER")


def _account_sbatch_args():
    return [f"--account={SLURM_ACCOUNT}"] if SLURM_ACCOUNT else []


def _mail_sbatch_args():
    return (["--mail-type=BEGIN,END,FAIL", f"--mail-user={SLURM_MAIL_USER}"]
            if SLURM_MAIL_USER else [])


# --- Container image (build target + what the benchmark runs against) ---
# Use the freshly-rebuilt image (adds the Python progress heartbeat + fast-tools-first
# combo order). It is a SEPARATE file from container/fastotf2-bench.sif, which a
# previously-launched job may still be using -- so pointing here does not disturb it.
IMAGE       = REPO_DIR / "container" / "fastotf2-bench-next.sif"
BASE_IMAGE  = "ghcr.io/hpc-ai-adv-dev/fastotf2/fastotf2-converter:latest"  # portable, single-locale

# --- Per-system configuration ------------------------------------------------
# trace_inputs: a dict {traced_nodes: path} (same shape as the scaling notebook), where each
#   path is a directory that directly contains the OTF2 anchor (traces.otf2). Node count is
#   only an input key -- every run/result is still labelled by the trace's real on-disk size
#   in GiB (measured with du -sb). NB: the Frontier HPL captures don't share one layout --
#   the 2/4/8-node anchors sit at the top level of their dir, but the 16/32-node ones live in
#   a results/ subdir, so those keys point at .../<run>/results (the dir holding traces.otf2).
# partition: blank -> the cluster's default (Frontier's is `batch`).
# chpl_threads: Chapel threads/locale; 0 = all node cores.
# sbatch_extra_args: account + mail (from secrets) on Frontier; none elsewhere.
SYSTEM_CONFIGS = {
    "frontier": {
        "trace_inputs": {
            2:   "/lustre/orion/csc688/world-shared/scorep-amd/runs/scratch/4098287/frontier-2-node-single-HPL-run",
            4:   "/lustre/orion/csc688/world-shared/scorep-amd/runs/scratch/4098294/frontier-4-node-single-HPL-run",
            8:   "/lustre/orion/csc688/world-shared/scorep-amd/runs/scratch/4098296/frontier-8-node-single-HPL-run",
            16:  "/lustre/orion/csc688/world-shared/scorep-amd/runs/frontier-16-node-single-HPL-run/results",
            32:  "/lustre/orion/csc688/world-shared/scorep-amd/runs/frontier-32-node-single-HPL-run/results",
            128: "/lustre/orion/csc688/world-shared/scorep-amd/runs/scorep-rocHPL/experiments/one-cabinet",
            384: "/lustre/orion/csc688/world-shared/scorep-amd/runs/scorep-rocHPL/experiments/three-cabinet",
        },
        "partition":         "",     # blank -> Frontier's default (batch)
        "chpl_threads":      56,
        "sbatch_extra_args": _account_sbatch_args() + _mail_sbatch_args(),
    },
    "other-ex": {
        "trace_inputs": {
            2:   "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-2-node-single-HPL-run",
            4:   "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-4-node-single-HPL-run",
            8:   "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-8-node-single-HPL-run",
            16:  "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-16-node-single-HPL-run",
            32:  "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-32-node-single-HPL-run",
            128: "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-128-node-single-HPL-run",
            384: "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-384-node-single-HPL-run",
        },
        "partition":         "",
        "chpl_threads":      0,
        "sbatch_extra_args": [],
    },
}

_cfg = SYSTEM_CONFIGS[SYSTEM]
if SYSTEM == "frontier" and not (SLURM_ACCOUNT and SLURM_MAIL_USER):
    raise RuntimeError(
        "SYSTEM='frontier' needs SLURM_ACCOUNT and SLURM_MAIL_USER -- create a gitignored "
        "local_secrets.py in the repo root with those two variables, or set them as "
        "environment variables before the Jupyter server process itself is started."
    )

TRACE_INPUTS = _cfg["trace_inputs"]   # {traced_nodes: trace directory}
TRACE_ANCHOR = "traces.otf2"

# Which traces (by their capture node count) to actually benchmark this run. Default: the
# ~0.7 GiB (2-node) and ~11.6 GiB (16-node) traces. The ~32.7 GiB (32-node) trace and larger
# ones are left out so this run neither duplicates nor interferes with a separately-running
# larger job -- add their node counts here to include them.
TRACES_TO_INCLUDE = [2, 16]

# --- Which (tool, format) combos to run. Each becomes its OWN SLURM job; jobs are
#     dependency-chained in §2 so they run one-at-a-time (clean, non-contending timings). ---
COMBOS = [
    ("chapel", "CSV"), ("chapel", "PARQUET"),
    ("c",      "CSV"),
    ("python", "CSV"), ("python", "PARQUET"),
]

REPEATS      = 1                   # timed repeats per combo (averaged in analysis)
CHPL_THREADS = _cfg["chpl_threads"]  # Chapel threads/locale; 0 = all node cores

# --- SLURM ---
SLURM_PARTITION   = _cfg["partition"]     # blank -> cluster default
SLURM_TIME        = "12:00:00"
SLURM_EXCLUSIVE   = True
SBATCH_EXTRA_ARGS = _cfg["sbatch_extra_args"]   # account + mail (from secrets) on Frontier

# --- Preview without submitting? ---
DRY_RUN = True
# ====================================================================

# --- Label each included trace by its measured on-disk size in GiB (du -sb, 1024**3) ---
_GIB = 1024 ** 3
def measure_gib(path):
    out = subprocess.run(["du", "-sb", path], capture_output=True, text=True, check=True)
    return int(out.stdout.split()[0]) / _GIB

TRACES = {}          # size label (e.g. "11.6GiB") -> trace directory
_trace_rows = []     # (traced_nodes, size label, path) -- for the readout below
for _nodes in sorted(TRACES_TO_INCLUDE):
    _p = TRACE_INPUTS[_nodes]
    _lbl = f"{measure_gib(_p):.1f}GiB"
    TRACES[_lbl] = _p
    _trace_rows.append((_nodes, _lbl, _p))

# --- Fresh, timestamped run folder (never clobbers a previous run) ---
RUN_TAG = f"run_{datetime.now():%Y%m%d_%H%M%S}"
OUT_ROOT = REPO_DIR / "out"
RUN_DIR  = OUT_ROOT / RUN_TAG
for sub in ("slurm_logs", "run_logs", "timings", "scratch", "plots"):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
TIMINGS_DIR = RUN_DIR / "timings"      # one small CSV per conversion job
RESULTS_CSV = RUN_DIR / "results.csv"  # merged from timings/ after jobs finish

CONFIG = {
    "run_tag": RUN_TAG, "system": SYSTEM, "image": str(IMAGE), "base_image": BASE_IMAGE,
    "traces": TRACES, "traces_to_include": TRACES_TO_INCLUDE, "trace_anchor": TRACE_ANCHOR,
    "combos": [f"{t}|{f}" for t, f in COMBOS], "repeats": REPEATS,
    "chpl_threads": CHPL_THREADS,
    "slurm": {"account": SLURM_ACCOUNT, "partition": SLURM_PARTITION,
              "time": SLURM_TIME, "exclusive": SLURM_EXCLUSIVE,
              "extra_args": SBATCH_EXTRA_ARGS},
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))

print(f"SYSTEM  : {SYSTEM}")
print(f"RUN_TAG : {RUN_TAG}")
print(f"RUN_DIR : {RUN_DIR}")
print(f"IMAGE   : {IMAGE}")
print("traces  :")
for _nodes, _lbl, _p in _trace_rows:
    print(f"    s{_nodes:<3} {_lbl:>8}  {_p}")
print(f"combos  : {', '.join(f'{t}({f})' for t, f in COMBOS)}")
print(f"jobs    : {len(TRACES) * len(COMBOS) * REPEATS} (one SLURM job each, dependency-chained)")
print(f"DRY_RUN : {DRY_RUN}")


# 1. Container image

Builds the bench image if `IMAGE` does not exist yet (one-time, ~minutes). Uses the
cached base image; installs the Python stack and compiles the C converter inside.

In [ ]:
if IMAGE.exists():
    print("Bench image present:", IMAGE)
else:
    print("Bench image missing — building it now (one-time)...")
    env = {**os.environ, "BENCH_SIF": str(IMAGE),
           "BENCH_IMAGE_TAG": "localhost/fastotf2-bench:latest",
           "BENCH_BASE_IMAGE": BASE_IMAGE}
    subprocess.run(["bash", str(REPO_DIR / "container" / "build-bench-image.sh")],
                   cwd=REPO_DIR, env=env, check=True)
    print("Build complete:", IMAGE)

# 2. Build & submit the benchmark to SLURM (one job per conversion)

Each `(trace, tool, format, repeat)` is submitted as its **own exclusive** single-node SLURM
job, but the jobs are **dependency-chained** (`--dependency afterany:<prev>`) so they run
**one-at-a-time** — the same strategy as the scaling notebook. These conversions are
I/O-heavy, so running them in parallel would make them contend for the shared filesystem and
pollute each other's wall-clock timings; serial execution keeps every measurement clean.
Every job writes its own one-row timing CSV under `timings/`, and its generated
`run_logs/<tag>.sbatch` is saved for provenance. With `DRY_RUN = True`, scripts are written
and the exact chained `sbatch` commands are printed, but nothing is submitted.


In [ ]:
import itertools

def sbatch_for(trace_label, trace_dir, tool, fmt, rep):
    """Text of a per-conversion sbatch script (config baked in for provenance)."""
    tag = f"{trace_label}_{tool}_{fmt}_r{rep}"
    result_file = TIMINGS_DIR / f"{tag}.csv"
    out_dir     = RUN_DIR / "scratch" / tag
    lines = ["#!/usr/bin/env bash",
             f"#SBATCH --job-name=f2b-{RUN_TAG}-{tag}",
             "#SBATCH --nodes=1",
             f"#SBATCH --time={SLURM_TIME}",
             f"#SBATCH --output={RUN_DIR}/slurm_logs/{tag}-%j.out",
             f"#SBATCH --error={RUN_DIR}/slurm_logs/{tag}-%j.err"]
    if SLURM_EXCLUSIVE:
        lines.append("#SBATCH --exclusive")
    if SLURM_PARTITION:
        lines.append(f"#SBATCH --partition={SLURM_PARTITION}")
    # Account + mail (from local_secrets, via SYSTEM_CONFIGS) are baked in as #SBATCH
    # directives so each saved script is a faithful record of what was submitted.
    for _extra in SBATCH_EXTRA_ARGS:
        lines.append(f"#SBATCH {_extra}")
    lines += [
        "",
        f"export BENCH_SIF={shlex.quote(str(IMAGE))}",
        f"export BENCH_RUN_TAG={shlex.quote(RUN_TAG)}",
        f"export BENCH_TRACE_DIR={shlex.quote(trace_dir)}",
        f"export BENCH_TRACE_LABEL={shlex.quote(trace_label)}",
        f"export BENCH_TOOL={shlex.quote(tool)}",
        f"export BENCH_FORMAT={shlex.quote(fmt)}",
        f"export BENCH_REPEAT={rep}",
        f"export BENCH_OUTPUT_DIR={shlex.quote(str(out_dir))}",
        f"export BENCH_RESULT_FILE={shlex.quote(str(result_file))}",
        f"export BENCH_TRACE_ANCHOR={shlex.quote(TRACE_ANCHOR)}",
        f"export BENCH_CHPL_THREADS={CHPL_THREADS}",
        "",
        f"bash {shlex.quote(str(REPO_DIR / 'benchmark' / 'run_one.sh'))}",
        "",
    ]
    return tag, "\n".join(lines)


# Every conversion is its own single-node job, but the jobs are DEPENDENCY-CHAINED
# (--dependency afterany:<prev>) so they run strictly one-at-a-time -- same strategy as the
# scaling notebook. These conversions are I/O-heavy, so letting them run in parallel would
# make them contend for the shared filesystem and pollute each other's wall-clock timings;
# serial execution keeps every measurement clean.
jobs, manifest = [], []
prev_job_id = None
for (label, path), (tool, fmt), rep in itertools.product(
        TRACES.items(), COMBOS, range(1, REPEATS + 1)):
    tag, script = sbatch_for(label, path, tool, fmt, rep)
    sb_path = RUN_DIR / "run_logs" / f"{tag}.sbatch"
    sb_path.write_text(script)

    sbatch_cmd = ["sbatch", "--parsable"]
    if prev_job_id is not None:
        sbatch_cmd += ["--dependency", f"afterany:{prev_job_id}"]
    sbatch_cmd.append(str(sb_path))

    if DRY_RUN:
        jobs.append({"tag": tag, "job_id": None})
        dep = f" (after {prev_job_id})" if prev_job_id is not None else " (chain head)"
        print(f"[dry] would submit {tag}{dep}\n        {' '.join(sbatch_cmd)}")
        prev_job_id = f"<{tag}>"   # keep the chain visible in the preview
        continue

    res = subprocess.run(sbatch_cmd, cwd=REPO_DIR, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"sbatch failed for {tag}:\n{res.stderr}")
    jid = res.stdout.strip().split(";")[0]
    jobs.append({"tag": tag, "job_id": jid})
    manifest.append(f"{jid},{tag}")
    print(f"submitted {jid}  {tag}" + (f"  (after {prev_job_id})" if prev_job_id else ""))
    prev_job_id = jid

JOB_IDS = [j["job_id"] for j in jobs if j["job_id"]]
(RUN_DIR / "manifest.csv").write_text("job_id,tag\n" + "".join(m + "\n" for m in manifest))
if DRY_RUN:
    print("\nDRY_RUN=True — nothing submitted. Per-job scripts written under run_logs/.")
else:
    print(f"\nSubmitted {len(JOB_IDS)} dependency-chained jobs for run {RUN_TAG} "
          "(they run one-at-a-time).")
    print(f"Manifest: {RUN_DIR / 'manifest.csv'}")


# 3. Monitor the queue (optional)

Jobs run one-at-a-time via the dependency chain, so timing CSVs appear one-by-one under
`timings/`. The widget below is a live `squeue` monitor (press **Start**): RUNNING jobs show
time-left to walltime, WAITING jobs show their estimated start + priority, and **Force Kill**
runs `scancel --me`. Or call `wait_until_my_jobs_finished()` to block until the chain is
done. Nothing here is required for the analysis — §4 reads whatever CSVs already exist.


In [ ]:
from workflows import watch_queue_widget, wait_until_my_jobs_finished

# Live squeue monitor (press Start): RUNNING jobs with time-left, WAITING jobs with their
# estimated start + priority, and a Force Kill (scancel --me). Safe to skip -- nothing here
# is required for the analysis, which reads whatever timing CSVs already exist.
watch_queue_widget()

# Or, to block this cell until the whole dependency chain has drained, run:
# wait_until_my_jobs_finished()


# 4. Results analysis

**Self-contained** — needs only a run's `results.csv`, so you can restart the kernel and run
just this section.

- `ANALYZE_RUN = None` → the run created above this session; if the kernel is fresh, the
  most recent run under `out/`.
- `ANALYZE_RUN = "run_YYYYMMDD_HHMMSS"` (a tag) or a full path → analyse that **previous**
  run without collecting new data.

In [ ]:
import re
from pathlib import Path
import pandas as pd
from plotnine import *  # noqa: F401,F403

# === Which run to analyse? ===
# None -> the run created above this session (or the latest under out/ on a fresh kernel).
# tag  -> a previous run's tag, e.g. "run_20260714_101500", or a full path.
ANALYZE_RUN = None

_OUT_ROOT = (REPO_DIR / "out") if "REPO_DIR" in globals() else (Path.cwd() / "out")

def resolve_run_dir(analyze_run):
    if analyze_run:
        p = Path(analyze_run)
        if not p.exists():
            p = _OUT_ROOT / analyze_run
        return p.parent if p.is_file() else p
    if "RUN_DIR" in globals():
        return Path(globals()["RUN_DIR"])
    runs = sorted(_OUT_ROOT.glob("run_*/timings"))
    if not runs:
        raise FileNotFoundError(f"No runs with timings/ under {_OUT_ROOT}")
    return runs[-1].parent

ANALYSIS_DIR = resolve_run_dir(ANALYZE_RUN)
PLOTS_DIR = ANALYSIS_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

tfiles = sorted((ANALYSIS_DIR / "timings").glob("*.csv"))
if not tfiles:
    raise FileNotFoundError(f"No per-job timing CSVs under {ANALYSIS_DIR / 'timings'}")
df = pd.concat([pd.read_csv(f) for f in tfiles], ignore_index=True)
print(f"Analysing run: {ANALYSIS_DIR.name}  ({len(df)} rows from {len(tfiles)} jobs)")
if (df["status"] != 0).any():
    print("WARNING: some conversions failed (status != 0):")
    print(df[df["status"] != 0][["trace", "tool", "format", "status"]].to_string(index=False))

ok = df[df["status"] == 0].copy()
agg = (ok.groupby(["trace", "tool", "format"], as_index=False)
         .agg(seconds=("seconds", "mean"), output_bytes=("output_bytes", "mean")))
agg["gib"] = agg["trace"].str.extract(r"([\d.]+)").astype(float)
agg["combo"] = agg["tool"].str.capitalize() + " (" + agg["format"] + ")"

# Ordered categoricals: traces ascending by real size; combos in a fixed reading order.
_trace_order = [t for _, t in sorted(set(zip(agg["gib"], agg["trace"])))]
agg["trace"] = pd.Categorical(agg["trace"], categories=_trace_order, ordered=True)
_combo_order = [c for c in ["Chapel (CSV)", "Chapel (PARQUET)", "C (CSV)",
                            "Python (CSV)", "Python (PARQUET)"] if c in set(agg["combo"])]
agg["combo"] = pd.Categorical(agg["combo"], categories=_combo_order, ordered=True)

# Save figures next to the run (kept alongside its data for provenance).
def _maybe_save(p, name, save=True):
    if save:
        p.save(PLOTS_DIR / f"{name}.png", dpi=150, verbose=False)

# Wide table for a quick numeric read-out (rows sorted by trace size).
table = (agg.pivot(index="trace", columns="combo", values="seconds").reindex(columns=_combo_order))
table.columns.name = None
table.index.name = "trace (size)"
print("\nConversion time (seconds), mean over repeats:")
table.round(2)


In [ ]:
# Speedup over the Python baseline, per format (tidy long form for plotnine).
_recs = []
_piv = agg.pivot_table(index=["trace", "gib"], columns=["format", "tool"],
                       values="seconds", observed=True)
for (trace, gib), row in _piv.iterrows():
    for fmt in row.index.get_level_values(0).unique():
        block = row[fmt]
        if "python" not in block or pd.isna(block.get("python")):
            continue
        for tool in block.index:
            if tool == "python" or pd.isna(block[tool]):
                continue
            _recs.append({"trace": trace, "gib": gib,
                          "comparison": f"{tool.capitalize()} ({fmt})",
                          "speedup": block["python"] / block[tool]})

speedup_long = pd.DataFrame(_recs)
if not speedup_long.empty:
    speedup_long["trace"] = pd.Categorical(speedup_long["trace"],
                                           categories=_trace_order, ordered=True)
    speedup_long = speedup_long.sort_values(["gib", "comparison"])
    print("Speedup over Python (x faster):")
    display(speedup_long.pivot(index="trace", columns="comparison", values="speedup").round(1))
else:
    print("No Python baseline available yet — speedup will appear once Python rows land.")


In [ ]:
# --- Graphs (plotnine), following the scaling-notebook conventions ---
# G1: conversion time vs trace size. Times span orders of magnitude (Chapel seconds vs
# Python thousands of seconds), so a log10 y-axis is used with points + connecting lines
# (a trend across data size) rather than bars, which keeps the log axis honest.
p_time = (
    ggplot(agg, aes("trace", "seconds", color="combo", group="combo"))
    + geom_line()
    + geom_point(size=3)
    + scale_y_log10()
    + scale_color_brewer(type="qual", palette="Dark2")
    + labs(title="OTF2 conversion time vs trace size",
           x="trace size", y="conversion time (s, log10)", color="tool (format)")
    + theme_bw()
    + theme(figure_size=(9, 5))
)
display(p_time)
_maybe_save(p_time, "g1_time")

# G2: speedup over Python -- a ratio, so bars from y=0 (the reference's rule: y-from-0 for
# magnitude/ratio bars; log axes only for scaling/decay curves).
if not speedup_long.empty:
    p_speedup = (
        ggplot(speedup_long, aes("trace", "speedup", fill="comparison"))
        + geom_col(position=position_dodge(width=0.8), width=0.7)
        + expand_limits(y=0)
        + scale_fill_brewer(type="qual", palette="Dark2")
        + labs(title="Speedup over Python (higher = faster)",
               x="trace size", y="x faster than Python", fill="tool (format)")
        + theme_bw()
        + theme(figure_size=(9, 5))
    )
    display(p_speedup)
    _maybe_save(p_speedup, "g2_speedup")
else:
    print("No Python baseline rows yet — speedup graph skipped.")


In [ ]:
# Persist merged results + a plain-text summary into the run folder (no extra deps).
results_out = ANALYSIS_DIR / "results.csv"
df.to_csv(results_out, index=False)

summary_txt = PLOTS_DIR / "benchmark_summary.txt"
with open(summary_txt, "w") as fh:
    fh.write(f"FastOTF2 converter benchmark ({ANALYSIS_DIR.name})\n\n")
    fh.write("Conversion time in seconds (mean over repeats):\n\n")
    fh.write(table.round(2).to_string())
    if not speedup_long.empty:
        fh.write("\n\nSpeedup over Python (x faster):\n\n")
        fh.write(speedup_long.pivot(index="trace", columns="comparison",
                                    values="speedup").round(1).to_string())
    fh.write("\n")

print("Wrote:", results_out)
print("Wrote:", summary_txt)
print("Figures:", PLOTS_DIR / "g1_time.png", "and", PLOTS_DIR / "g2_speedup.png")
print("\n" + summary_txt.read_text())


## Methodology & tips

- Each conversion runs as its **own exclusive single-node SLURM job**, and the jobs are
  **dependency-chained** so they run **one-at-a-time** — these conversions are I/O-heavy, so
  serial execution avoids shared-filesystem contention that would pollute the wall-clock
  timings. Wall-clock time is measured around each converter invocation; output goes to that
  job's `scratch/<tag>` and is deleted afterwards.
- Chapel uses all node cores (`CHPL_RT_NUM_THREADS_PER_LOCALE`) — its data-parallel
  advantage. Python and C run single-threaded, as shipped. C is CSV-only (Parquet in C needs
  the Arrow C++/GLib toolchain, out of scope).
- Graphs use **plotnine** (`theme_bw`, Brewer *Dark2*, ordered size categoricals): a log10
  axis for the multi-order-of-magnitude conversion **times**, and y-from-0 bars for the
  **speedup** ratio.

**Runs & re-analysis:** every run lives under `out/run_<timestamp>/` with its own
`timings/` (one CSV per job), merged `results.csv`, `slurm_logs/`, `run_logs/` (the exact
per-job `.sbatch`), `config.json`, `manifest.csv`, and `plots/`. To study an old run,
restart the kernel, run the §0 imports, set `ANALYZE_RUN` in §4 to the run tag, and run the
analysis cells — no resubmission.

**Combining runs (e.g. this 16-GiB run + a separate 32-GiB run):** point the §4 loader at
multiple `timings/` folders, or copy per-job CSVs together — since every row is self-
describing (`run_tag, trace, tool, format, ...`), concatenating them Just Works.

**Testing a rebuilt image without disturbing a running job:** build to a separate file and
point `IMAGE` at it, e.g. `BENCH_SIF=container/fastotf2-bench-next.sif bash
container/build-bench-image.sh --force`.
